# Desktop Voice Orchestrator — M3 个性化 ASR 扩词与抗噪精调流水线 (`asr-personal-v2`, Colab A100 加速版)

本 Notebook 在 **Google Colab (推荐 A100 GPU)** 上一键完成 M3 阶段全流程，并内置 **`/content/local_data` 本地 NVMe 高速暂存 + 云盘增量断点同步**（彻底规避 Google Drive FUSE 读写数千个 `.wav` 小文件的 IO 卡顿）：
1. **Cell 1**：挂载 Google Drive，检查 A100 GPU，拉取最新 `develop` 分支代码，并将云盘数据集高速同步至本地盘 `/content/local_data`；
2. **Cell 2**：安装环境依赖（`CosyVoice2`、`sherpa-onnx`、`funasr`、`kaldi-native-fbank`、`onnxruntime-gpu`、`faster-whisper`、`pypinyin`）并拉取三大基础模型；
3. **Cell 3（先试听把关）**：在本地高速盘运行 `--mode pilot` 生成 **25 条** 个人音色克隆试听小样（仅用于喂给 ASR 训练新歌名与中英混杂词），直接在页面内渲染播放器与独立裁判质检报告；
4. **Cell 4（清洗双人连麦与笑声会话）**：自动将云盘 `game_sessions/`（166 分钟双人对话 + 高频笑声）拷入本地盘，执行 Silero VAD 切分 + `SenseVoice / Whisper large-v3` 双模型 100% 逐字一致正样本提取与笑声防误唤醒负样本分流，完成后归档回云盘；
5. **Cell 5（批量扩词造数 + 自动断点备份）**：在本地高速盘运行全量 **3,600 条** 克隆合成 + 独立裁判闭环过滤（后台每 2 分钟自动增量同步回云盘，支持 `--resume` 断点续跑）；
6. **Cell 6（A100 精调与导出 `asr-personal-v2`）**：以本人真机录音为主锚点（`--repeat 4`）+ 动态抽样合成语料（`--tts-max-per-epoch 1800`）+ `0~15 dB` 真实音乐混音，解冻编码器后 3 层微调并导出 `QUInt8` `model.int8.onnx` 到云盘 `models/asr-personal-v2/`，输出完整多维对比评测表。

## Cell 1：挂载 Google Drive、拉取 `develop` 分支并同步数据至 `/content/local_data` 本地高速盘

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import drive

# =============================================================================
# 运行时控制开关
# =============================================================================
ENABLE_REMOTE_RESCUE = True
AUTO_DISCONNECT_A100 = True

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/desktop-voice-orchestrator-data")
LOCAL_DATA = Path("/content/local_data")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/desktop-voice-orchestrator")
REPO_URL = "https://github.com/CICSilver/desktop-voice-orchestrator.git"
REPO_BRANCH = "develop"

print("=== 1. 准备最新训练脚本仓库 (develop 分支) ===")
if not REPO_DIR.is_dir():
    print(f"正在克隆仓库 {REPO_URL} ({REPO_BRANCH}) ...")
    rc = subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)]).returncode
    if rc != 0:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

# Overlay repo_sync hotfixes if present on Drive
DRIVE_REPO_SYNC = DRIVE_ROOT / "repo_sync"
if DRIVE_REPO_SYNC.is_dir():
    print(f"🔄 检测到云盘热补丁目录: {DRIVE_REPO_SYNC}，正在覆盖同步到本地 NVMe {REPO_DIR} ...")
    subprocess.run(["rsync", "-a", f"{DRIVE_REPO_SYNC}/", f"{REPO_DIR}/"], check=True)
    print(f"✅ repo_sync 热补丁已成功覆盖应用到: {REPO_DIR}")

print(f"当前代码工作区路径: {REPO_DIR}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from training.colab_telemetry_hook import ColabJobHook

hook = ColabJobHook(
    "m3_asr",
    drive_data_root=DRIVE_ROOT,
    repo_dir=REPO_DIR,
    total_stages=6,
    rescue_wait_sec=900.0 if ENABLE_REMOTE_RESCUE else 0.0,
    auto_disconnect=AUTO_DISCONNECT_A100,
)

def stage1_setup():
    print("=== GPU 硬件检查 ===")
    gpu_info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    print(gpu_info.stdout.strip() if gpu_info.returncode == 0 else "⚠️ 未检测到 NVIDIA GPU，请切换运行时到 A100（或 L4）！")

    print("\n=== 云盘数据目录完整性体检 ===")
    checks = {
        "云盘数据根目录": DRIVE_ROOT,
        "录音切片目录 (dataset/clips)": DRIVE_ROOT / "dataset" / "clips",
        "录音主清单 (dataset/manifest.jsonl)": DRIVE_ROOT / "dataset" / "manifest.jsonl",
        "本人黄金参考音频池 (prompt_pool.jsonl)": DRIVE_ROOT / "m3_tts_prep" / "prompt_pool" / "prompt_pool.jsonl",
        "25条试听合成清单 (pilot_prompts_25.jsonl)": DRIVE_ROOT / "m3_tts_prep" / "pilot_prompts_25.jsonl",
        "3600条全量扩词清单 (full_prompts_3600.jsonl)": DRIVE_ROOT / "m3_tts_prep" / "full_prompts_3600.jsonl",
        "整合训练数据包 (dvo_voice_training_data.tar.gz)": DRIVE_ROOT / "dvo_voice_training_data.tar.gz",
        "双人连麦与笑声会话目录 (game_sessions)": DRIVE_ROOT / "game_sessions",
    }
    for label, p in checks.items():
        exists = p.exists()
        mark = "✅ 就绪" if exists else "⚠️ 缺失 (请确认已完成 Google Drive 同步)"
        print(f"  {mark} | {label}: {p}")

    print("\n=== 同步核心数据集与已有断点产物至本地高速 SSD (/content/local_data) ===")
    stale_report_f = DRIVE_ROOT / "m3_tts_prep" / "pilot_audio" / "quality_report.json"
    if stale_report_f.is_file():
        try:
            import json as _json
            import shutil as _shutil
            stale_rep = _json.loads(stale_report_f.read_text("utf-8"))
            if stale_rep.get("accepted", 0) == 0:
                print(f"🧹 检测到云盘历史 0/25 残损 pilot_audio，正在清理防止污染本地缓存: {stale_report_f.parent}")
                _shutil.rmtree(stale_report_f.parent, ignore_errors=True)
        except Exception as _e:
            print(f"⚠️ 检查历史 pilot_audio 异常: {_e}")

    def rsync_dir(src: Path, dst: Path, desc: str) -> None:
        if not src.exists():
            return
        dst.parent.mkdir(parents=True, exist_ok=True)
        src_arg = f"{src}/" if src.is_dir() else str(src)
        dst_arg = f"{dst}/" if src.is_dir() else str(dst)
        if src.is_dir():
            dst.mkdir(parents=True, exist_ok=True)
        print(f"  -> 正在同步 {desc} ...")
        subprocess.run(["rsync", "-a", src_arg, dst_arg], check=True)

    ARCHIVE_TAR = DRIVE_ROOT / "dvo_voice_training_data.tar.gz"
    if ARCHIVE_TAR.is_file() and not (LOCAL_DATA / "dataset" / "manifest.jsonl").is_file():
        print("📦 检测到整合归档包 dvo_voice_training_data.tar.gz，正在顺序流式解压至本地高速 NVMe (/content/local_data) ...")
        subprocess.run(["tar", "-xzf", str(ARCHIVE_TAR), "-C", str(LOCAL_DATA)], check=True)
        # Overlay resumable Drive checkpoints if they exist
        rsync_dir(DRIVE_ROOT / "m3_tts_prep" / "synth_dataset_3600", LOCAL_DATA / "m3_tts_prep" / "synth_dataset_3600", "3600条合成断点缓存 (synth_dataset_3600)")
        rsync_dir(DRIVE_ROOT / "m3_tts_prep" / "game_mined", LOCAL_DATA / "m3_tts_prep" / "game_mined", "连麦会话清洗断点缓存 (game_mined)")
    else:
        if not (LOCAL_DATA / "dataset" / "manifest.jsonl").is_file():
            rsync_dir(DRIVE_ROOT / "dataset", LOCAL_DATA / "dataset", "主数据集 (dataset/)")
        rsync_dir(DRIVE_ROOT / "m3_tts_prep", LOCAL_DATA / "m3_tts_prep", "扩词参考池与断点缓存 (m3_tts_prep/)")
    print("✅ 本地高速盘预热完成！后续合成与微调将全程在 NVMe 本地盘高速运行。")
    hook.update_progress(message="stage1_completed", force_flush=True)

hook.run_stage("cell1_setup_and_stage_nvme", 1, stage1_setup)


## Cell 2：一键安装环境依赖并拉取三大基础模型

In [ ]:
import os
import subprocess
import tarfile
import urllib.request
from pathlib import Path

MODELS_ROOT = Path("/content/models")
MODELS_ROOT.mkdir(parents=True, exist_ok=True)
COSY_MODEL_DIR = MODELS_ROOT / "CosyVoice2-0.5B"
NANO_PRETRAINED_DIR = MODELS_ROOT / "Fun-ASR-Nano-2512"
JUDGE_MODEL_DIR = MODELS_ROOT / "sherpa-onnx-sense-voice-funasr-nano-int8-2025-12-17"
COSY_REPO = Path("/content/CosyVoice")
SHERPA_REPO = Path("/content/sherpa-onnx")

def stage2_install_deps_and_models():
    print("=== 1. 安装核心 Python 依赖 ===")
    pip_pkgs = [
        "sherpa-onnx",
        "funasr",
        "kaldi-native-fbank",
        "onnx",
        "onnxruntime-gpu",
        "pypinyin",
        "scipy",
        "soundfile",
        "librosa",
        "modelscope",
        "huggingface_hub",
        "faster-whisper",
        "hyperpyyaml",
        "conformer",
        "omegaconf",
        "hydra-core",
        "diffusers",
        "lightning",
        "inflect",
        "wetext",
        "pyworld-prebuilt",
        "tiktoken",
        "openai-whisper",
        "gdown",
        "einops",
        "wget",
        "rootutils",
        "Unidecode",
        "nvidia-cudnn-cu12",
        "nvidia-cublas-cu12",
    ]
    hook.run_cmd(["pip", "install", "-q", *pip_pkgs], stage_name="cell2_pip_install")

    print("=== 2. 准备 CosyVoice2 源码与 Matcha-TTS 子模块 ===")
    if not COSY_REPO.is_dir():
        hook.run_cmd([
            "git", "clone", "--depth", "1", "--recursive",
            "https://github.com/FunAudioLLM/CosyVoice.git", str(COSY_REPO)
        ], stage_name="cell2_cosy_clone")
    os.environ["COSYVOICE_REPO"] = str(COSY_REPO)

    # Python 3.13 pyworld soft-import safeguard for CosyVoice processor.py
    proc_file = COSY_REPO / "cosyvoice" / "dataset" / "processor.py"
    if proc_file.is_file():
        proc_code = proc_file.read_text("utf-8")
        if "\nimport pyworld as pw\n" in proc_code:
            proc_code = proc_code.replace(
                "\nimport pyworld as pw\n",
                "\ntry:\n    import pyworld as pw\nexcept ImportError:\n    pw = None\n",
            )
            proc_file.write_text(proc_code, "utf-8")
            print("✅ 已对 CosyVoice processor.py 应用 Python 3.13 pyworld 软导入防护。")

    print("=== 3. 准备 sherpa-onnx Fun-ASR-Nano 训练定义 (nano.py) ===")
    if not SHERPA_REPO.is_dir():
        hook.run_cmd([
            "git", "clone", "--depth", "1",
            "https://github.com/k2-fsa/sherpa-onnx.git", str(SHERPA_REPO)
        ], stage_name="cell2_sherpa_clone")
    os.environ["DVO_SHERPA_NANO_DIR"] = str(SHERPA_REPO / "scripts/sense-voice/rknn")

    print("=== 4. 下载 CosyVoice2-0.5B 与 Fun-ASR-Nano-2512 预训练权重 ===")
    from huggingface_hub import snapshot_download
    if not (COSY_MODEL_DIR / "cosyvoice2.yaml").is_file():
        try:
            snapshot_download("FunAudioLLM/CosyVoice2-0.5B", local_dir=str(COSY_MODEL_DIR))
        except Exception:
            from modelscope import snapshot_download as ms_download
            ms_download("iic/CosyVoice2-0.5B", local_dir=str(COSY_MODEL_DIR))

    if not (NANO_PRETRAINED_DIR / "model.pt").is_file():
        snapshot_download("FunAudioLLM/Fun-ASR-Nano-2512", local_dir=str(NANO_PRETRAINED_DIR))

    print("=== 5. 下载原始未微调 sherpa-onnx SenseVoice 独立裁判模型 & tokens.txt ===")
    if not (JUDGE_MODEL_DIR / "model.int8.onnx").is_file():
        tar_url = (
            "https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/"
            "sherpa-onnx-sense-voice-funasr-nano-int8-2025-12-17.tar.bz2"
        )
        tar_path = MODELS_ROOT / "sense-voice-nano.tar.bz2"
        urllib.request.urlretrieve(tar_url, str(tar_path))
        with tarfile.open(tar_path, "r:bz2") as tf:
            tf.extractall(MODELS_ROOT)
        tar_path.unlink(missing_ok=True)

    os.environ["DVO_SHERPA_TOKENS"] = str(JUDGE_MODEL_DIR / "tokens.txt")

    # Inject CUDA / cuDNN library paths into runtime environment
    from training.colab_telemetry_hook import inject_cuda_ld_library_path
    cuda_env = inject_cuda_ld_library_path()
    os.environ.update(cuda_env)
    print("CUDA/cuDNN LD_LIBRARY_PATH active:")
    print(" ", os.environ.get("LD_LIBRARY_PATH", "(unset)"))

    print("\n✅ 所有依赖与三大基础模型准备完毕！")
    print("  - CosyVoice2-0.5B :", COSY_MODEL_DIR)
    print("  - Fun-ASR-Nano    :", NANO_PRETRAINED_DIR)
    print("  - 独立裁判模型     :", JUDGE_MODEL_DIR)
    hook.update_progress(message="cell2_completed", force_flush=True)

hook.run_stage("cell2_install_deps_and_models", 2, stage2_install_deps_and_models)


## Cell 3（先试听把关）：本地高速盘合成 25 条 Pilot 试听小样 + 渲染播放器与质检报告

In [ ]:
import json
import subprocess
from pathlib import Path
from IPython.display import Audio, HTML, display

PILOT_OUT = LOCAL_DATA / "m3_tts_prep" / "pilot_audio"
DRIVE_PILOT_OUT = DRIVE_ROOT / "m3_tts_prep" / "pilot_audio"
PROMPT_POOL_JSONL = LOCAL_DATA / "m3_tts_prep" / "prompt_pool" / "prompt_pool.jsonl"
PILOT_PROMPTS_JSONL = LOCAL_DATA / "m3_tts_prep" / "pilot_prompts_25.jsonl"

def stage3_pilot_tts_25():
    cmd = [
        "python", str(REPO_DIR / "training/asr/synthesize_personal_tts.py"),
        "--mode", "pilot",
        "--prompts", str(PILOT_PROMPTS_JSONL),
        "--prompt-pool", str(PROMPT_POOL_JSONL),
        "--cosyvoice-model", str(COSY_MODEL_DIR),
        "--cosyvoice-repo", str(COSY_REPO),
        "--judge-model-dir", str(JUDGE_MODEL_DIR),
        "--out-dir", str(PILOT_OUT),
        "--personal-ratio", "1.0",
    ]
    hook.run_cmd(cmd, stage_name="cell3_pilot_tts")

    DRIVE_PILOT_OUT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-a", f"{PILOT_OUT}/", f"{DRIVE_PILOT_OUT}/"], check=True)

    report = json.loads((PILOT_OUT / "quality_report.json").read_text("utf-8"))
    print("\n📊 25 条 Pilot 闭环质检汇总:", json.dumps(report, ensure_ascii=False, indent=2))
    if report.get("accepted", 0) <= 0:
        raise RuntimeError(f"Pilot synthesis gate failed: 0 items accepted out of 25! Report: {report}")

    ref_first = json.loads(PROMPT_POOL_JSONL.read_text("utf-8").splitlines()[0])
    ref_wav = PROMPT_POOL_JSONL.parent / ref_first["wav_path"]
    display(HTML(f"<h4>🎧 本人原始参考音频对照（{ref_first['id']}）：<code>{ref_first['prompt_text']}</code></h4>"))
    display(Audio(filename=str(ref_wav)))

    display(HTML("<h4>🔊 CosyVoice2 零样本克隆 + SenseVoice 独立裁判通过样本试听（前 15 条）：</h4>"))
    manifest_lines = (PILOT_OUT / "manifest.jsonl").read_text("utf-8").splitlines()
    for line in manifest_lines[:15]:
        rec = json.loads(line)
        wav_file = PILOT_OUT / rec["audio"]
        display(HTML(
            f"<div style='margin-top:8px;padding:6px;border-left:3px solid #2b8a3e;background:#f8f9fa;'>"
            f"<b>[{rec['id']}]</b> 目标文本：<code>{rec['prompt']}</code><br/>"
            f"裁判识别：<code>{rec['judge_text']}</code> | CER={rec.get('char_cer', rec.get('judge_cer', 0.0))} | "
            f"语速={rec['chars_per_sec']} 字/秒 | 时长={rec['duration_s']}s | 参考源={rec['ref_prompt_id']} (第{rec['attempt']}次通过)"
            f"</div>"
        ))
        display(Audio(filename=str(wav_file)))
    hook.update_progress(message="cell3_completed", force_flush=True)

hook.run_stage("cell3_pilot_tts_25", 3, stage3_pilot_tts_25)


## Cell 4：清洗双人连麦与高频笑声录音（本地盘 VAD 切分 + 双模型交叉伪标注 + 自动归档）

In [ ]:
import json
import subprocess
from collections import Counter
from pathlib import Path

FORCE_REMINE = False  # 如追加了新的 game_sessions 会话可改为 True 重新切分

DRIVE_GAME_IN = DRIVE_ROOT / "game_sessions"
LOCAL_GAME_IN = LOCAL_DATA / "game_sessions"
LOCAL_GAME_OUT = LOCAL_DATA / "m3_tts_prep" / "game_mined"
DRIVE_GAME_OUT = DRIVE_ROOT / "m3_tts_prep" / "game_mined"

def stage4_mine_game_sessions():
    if (LOCAL_GAME_OUT / "manifest.jsonl").is_file() and not FORCE_REMINE:
        print(f"✅ 检测到已清洗完成的连麦/笑声数据集 {LOCAL_GAME_OUT / 'manifest.jsonl'}，直接复用（无需重复切分）。")
    elif DRIVE_GAME_IN.is_dir() and any(DRIVE_GAME_IN.iterdir()):
        print("📥 正在将云盘 game_sessions (约 1.2 GB) 顺序拷入本地高速 SSD ...")
        LOCAL_GAME_IN.mkdir(parents=True, exist_ok=True)
        subprocess.run(["rsync", "-a", "--info=progress2", f"{DRIVE_GAME_IN}/", f"{LOCAL_GAME_IN}/"], check=True)

        vad_path = REPO_DIR / "models/silero_vad.int8.onnx"
        if not vad_path.is_file() and (LOCAL_DATA / "models/silero_vad.int8.onnx").is_file():
            vad_path = LOCAL_DATA / "models/silero_vad.int8.onnx"
        if not vad_path.is_file() and (DRIVE_ROOT / "models/silero_vad.int8.onnx").is_file():
            vad_path = DRIVE_ROOT / "models/silero_vad.int8.onnx"

        print("🎮 启动本地高速盘 Silero VAD 切分 + SenseVoice / Whisper large-v3 双模型交叉校验...")
        hook.run_cmd([
            "python", str(REPO_DIR / "tools/mine_game_sessions.py"),
            "--sessions", str(LOCAL_GAME_IN),
            "--out", str(LOCAL_GAME_OUT),
            "--vad-model", str(vad_path),
            "--transcribe",
            "--sensevoice-dir", str(JUDGE_MODEL_DIR),
            "--whisper-model", "large-v3",
            "--device", "cuda",
        ], stage_name="cell4_mine_game_sessions")

        print("📤 正在将切分与标注结果归档回 Google Drive ...")
        DRIVE_GAME_OUT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["rsync", "-a", f"{LOCAL_GAME_OUT}/", f"{DRIVE_GAME_OUT}/"], check=True)
        print(f"✅ 已归档至: {DRIVE_GAME_OUT}")
    else:
        print(f"ℹ️ 未检测到 {DRIVE_GAME_IN}，跳过此步。")

    if (LOCAL_GAME_OUT / "manifest.jsonl").is_file():
        recs = [json.loads(x) for x in (LOCAL_GAME_OUT / "manifest.jsonl").read_text("utf-8").splitlines() if x.strip()]
        dist = Counter((r.get("split"), r.get("kind")) for r in recs)
        print("📊 连麦与笑声会话挖掘样本分布:", {f"{s}/{k}": n for (s, k), n in sorted(dist.items())})
    hook.update_progress(message="cell4_completed", force_flush=True)

hook.run_stage("cell4_mine_game_sessions", 4, stage4_mine_game_sessions)


## Cell 5（批量扩词造数）：本地高速盘 3,600 条克隆合成 + 后台每 2 分钟增量云盘断点保护

In [ ]:
import subprocess
import threading

FULL_PROMPTS_JSONL = LOCAL_DATA / "m3_tts_prep" / "full_prompts_3600.jsonl"
LOCAL_SYNTH_OUT = LOCAL_DATA / "m3_tts_prep" / "synth_dataset_3600"
DRIVE_SYNTH_OUT = DRIVE_ROOT / "m3_tts_prep" / "synth_dataset_3600"

LOCAL_SYNTH_OUT.mkdir(parents=True, exist_ok=True)
DRIVE_SYNTH_OUT.mkdir(parents=True, exist_ok=True)

def stage5_full_tts_3600():
    stop_sync = threading.Event()

    def periodic_drive_backup() -> None:
        while not stop_sync.wait(120.0):
            subprocess.run(["rsync", "-a", f"{LOCAL_SYNTH_OUT}/", f"{DRIVE_SYNTH_OUT}/"], check=False)

    sync_thread = threading.Thread(target=periodic_drive_backup, daemon=True)
    sync_thread.start()

    try:
        # --personal-ratio 0.80: 80% 本人音色克隆 + 20% 语速/辅助说话人多样化，防止 ASR 过拟合单一声码器指纹
        # --resume: 若中途断开，重跑会自动接着本地/云盘已有条目继续合成
        hook.run_cmd([
            "python", str(REPO_DIR / "training/asr/synthesize_personal_tts.py"),
            "--mode", "full",
            "--resume",
            "--prompts", str(FULL_PROMPTS_JSONL),
            "--prompt-pool", str(PROMPT_POOL_JSONL),
            "--personal-ratio", "0.80",
            "--cosyvoice-model", str(COSY_MODEL_DIR),
            "--cosyvoice-repo", str(COSY_REPO),
            "--judge-model-dir", str(JUDGE_MODEL_DIR),
            "--out-dir", str(LOCAL_SYNTH_OUT),
        ], stage_name="cell5_full_tts")
    finally:
        stop_sync.set()
        sync_thread.join(timeout=10.0)
        print("📤 正在执行最终增量同步至 Google Drive ...")
        subprocess.run(["rsync", "-a", f"{LOCAL_SYNTH_OUT}/", f"{DRIVE_SYNTH_OUT}/"], check=True)
        print(f"✅ 3,600 条扩词合成语料已完整落盘备份至: {DRIVE_SYNTH_OUT}")
    hook.update_progress(message="cell5_completed", force_flush=True)

hook.run_stage("cell5_full_tts_3600", 5, stage5_full_tts_3600)


## Cell 6：A100 精调与导出 `asr-personal-v2`（本地高速盘读取 + 动态音乐混音 + INT8 导出对比）

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

DATASET_DIR = LOCAL_DATA / "dataset"
assert (DATASET_DIR / "manifest.jsonl").is_file(), (
    f"未找到 {DATASET_DIR / 'manifest.jsonl'}！请先运行 Cell 1 确认云盘 dataset/manifest.jsonl 已同步。"
)

EXP_DIR = Path("/content/exp/asr-personal-v2")
FINAL_EXPORT_DRIVE = DRIVE_ROOT / "models" / "asr-personal-v2"

def stage6_finetune_export_eval():
    extra_manifests = []
    synth_mf = LOCAL_DATA / "m3_tts_prep" / "synth_dataset_3600" / "manifest.jsonl"
    if synth_mf.is_file():
        extra_manifests.append(str(synth_mf))

    game_mf = LOCAL_DATA / "m3_tts_prep" / "game_mined" / "manifest.jsonl"
    if game_mf.is_file():
        extra_manifests.append(str(game_mf))

    print("=== 1. 启动 Fun-ASR-Nano CTC 个性化精调 (A100 高速本地盘) ===")
    print("附加语料清单:", extra_manifests)
    train_cmd = [
        "python", str(REPO_DIR / "training/asr/finetune.py"),
        "--dataset", str(DATASET_DIR),
        "--pretrained", str(NANO_PRETRAINED_DIR),
        "--sherpa-nano-dir", os.environ["DVO_SHERPA_NANO_DIR"],
        "--tokens", os.environ["DVO_SHERPA_TOKENS"],
        "--aishell-utterances", "0",
        "--repeat", "4",
        "--tts-max-per-epoch", "1800",
        "--num-epochs", "4",
        "--lr", "8e-5",
        "--unfreeze-encoder-layers", "3",
        "--encoder-lr", "1.5e-5",
        "--mix-probability", "0.5",
        "--snr", "0", "15",
        "--exp-dir", str(EXP_DIR),
    ]
    if extra_manifests:
        train_cmd += ["--extra-manifests", *extra_manifests]

    hook.run_cmd(train_cmd, stage_name="cell6_finetune")

    print("\n=== 2. 逐轮导出 QUInt8 ONNX 并运行多维离线评测对比 ===")
    extra_test_mfs = [str(game_mf)] if game_mf.is_file() else []
    new_cmd_mf = DRIVE_ROOT / "experiments/2026-11-10-openai-vs-local-eval/clips_20260926_225120/manifest.jsonl"

    def run_eval(model_dir: Path, out_json: Path) -> dict:
        cmd = [
            "python", str(REPO_DIR / "training/asr/evaluate.py"),
            "--model-dir", str(model_dir),
            "--dataset", str(DATASET_DIR),
            "--out", str(out_json),
        ]
        if extra_test_mfs:
            cmd += ["--extra-test-manifests", *extra_test_mfs]
        if new_cmd_mf.is_file():
            cmd += ["--new-commands-manifest", str(new_cmd_mf)]
        hook.run_cmd(cmd, stage_name="cell6_evaluate")
        return json.loads(out_json.read_text("utf-8"))

    print("--- Baseline (Original SenseVoice-Nano) ---")
    base_rep = run_eval(JUDGE_MODEL_DIR, EXP_DIR / "eval-baseline-orig.json")

    rows = [("Baseline (Orig)", base_rep)]
    best_epoch = None
    best_score = 1e9

    for epoch in range(1, 5):
        ckpt = EXP_DIR / f"epoch-{epoch}.pt"
        if not ckpt.is_file():
            continue
        export_dir = EXP_DIR / f"export-epoch-{epoch}"
        hook.run_cmd([
            "python", str(REPO_DIR / "training/asr/export.py"),
            "--checkpoint", str(ckpt),
            "--out", str(export_dir),
            "--sherpa-nano-dir", os.environ["DVO_SHERPA_NANO_DIR"],
            "--tokens", os.environ["DVO_SHERPA_TOKENS"],
            "--comment", f"Fun-ASR-Nano-2512 CTC branch, asr-personal-v2 epoch {epoch}",
        ], stage_name=f"cell6_export_epoch_{epoch}")

        print(f"--- Epoch {epoch} ---")
        rep = run_eval(export_dir, EXP_DIR / f"eval-epoch-{epoch}.json")
        rows.append((f"Epoch {epoch}", rep))

        cer_q = rep["cer"].get("quiet", 0.0)
        cer_m = rep["cer"].get("music", 0.0)
        cer_read = rep["cer"].get("dev_read", 0.0)
        bg_fp = rep.get("background_clips_with_command_phrase", 0)
        composite = cer_q + cer_m + cer_read * 0.25 + bg_fp * 0.5
        if composite < best_score:
            best_score = composite
            best_epoch = epoch

    print("\n================ 📊 评测对比总表 ================")
    print(f"{'模型版本':<18} | {'CER(安静)':<10} | {'CER(外放音乐)':<12} | {'CER(通用朗读)':<12} | {'背景/笑声误出命令词'}")
    print("-" * 78)
    for title, r in rows:
        cq = f"{r['cer'].get('quiet', 0)*100:.2f}%"
        cm = f"{r['cer'].get('music', 0)*100:.2f}%"
        cr = f"{r['cer'].get('dev_read', 0)*100:.2f}%"
        bf = f"{r.get('background_clips_with_command_phrase', 0)}/{r.get('background_clips', 0)}"
        print(f"{title:<18} | {cq:<10} | {cm:<12} | {cr:<12} | {bf}")

    assert best_epoch is not None, "未找到导出的训练轮次检查点！"
    best_export = EXP_DIR / f"export-epoch-{best_epoch}"
    FINAL_EXPORT_DRIVE.mkdir(parents=True, exist_ok=True)
    for fname in ["model.int8.onnx", "model.onnx", "tokens.txt"]:
        src_f = best_export / fname
        if src_f.is_file():
            shutil.copy2(src_f, FINAL_EXPORT_DRIVE / fname)
    shutil.copy2(EXP_DIR / "summary.json", FINAL_EXPORT_DRIVE / "train_summary.json")
    shutil.copy2(EXP_DIR / f"eval-epoch-{best_epoch}.json", FINAL_EXPORT_DRIVE / "eval_report.json")

    print(f"\n🎉 已自动选出最优轮次 Epoch {best_epoch} 并同步至 Google Drive:")
    print(f"   {FINAL_EXPORT_DRIVE}")
    print("   回主力机只需将 `model.int8.onnx` 与 `tokens.txt` 放入 `models/asr-personal-v2/` 并更新 `config/local.toml` 即可生效！")

    summary_metrics = {
        "best_epoch": best_epoch,
        "best_composite_score": round(best_score, 4),
        "best_cer": rows[best_epoch][1]["cer"] if best_epoch < len(rows) else {},
    }
    hook.finish_job(
        artifacts=[
            FINAL_EXPORT_DRIVE / "model.int8.onnx",
            FINAL_EXPORT_DRIVE / "eval_report.json",
            FINAL_EXPORT_DRIVE / "train_summary.json",
        ],
        summary_metrics=summary_metrics,
    )

hook.run_stage("cell6_finetune_export_eval", 6, stage6_finetune_export_eval)
